# Preprocess BPA transmission line interruptions

Extract the `Transmission Line Interruptions` table from every BPA workbook, concatenate the tables, and save the result as `bpa_processed.csv`.

In [1]:
from pathlib import Path

import pandas as pd

In [ ]:
BPA_DATA_DIR = Path("data/bpa")
OUTPUT_PATH = Path("bpa_concatenated.csv")
TABLE_TITLE = "Transmission Line Interruptions"

In [3]:
def extract_transmission_line_tables(workbook_path: Path) -> list[pd.DataFrame]:
    """Return all Transmission Line Interruptions tables in one workbook."""
    raw = pd.read_excel(workbook_path, sheet_name=0, header=None)

    # Section headings occupy a row by themselves and contain 'Interruptions'.
    section_rows = [
        index
        for index, row in raw.iterrows()
        if row.notna().sum() == 1
        and "Interruptions" in str(row.dropna().iloc[0]).strip()
    ]
    title_rows = [
        index
        for index in section_rows
        if str(raw.loc[index].dropna().iloc[0]).strip() == TABLE_TITLE
    ]

    tables = []
    for title_row in title_rows:
        header_row = title_row + 1
        later_sections = [index for index in section_rows if index > title_row]
        end_row = min(later_sections, default=len(raw))

        headers = [
            str(value).strip() if pd.notna(value) else f"unnamed_{column}"
            for column, value in enumerate(raw.loc[header_row])
        ]
        table = raw.iloc[header_row + 1 : end_row].copy()
        table.columns = headers
        table = table.dropna(how="all")

        # Data rows begin with an outage datetime; this removes notes and footers.
        outage_datetime = pd.to_datetime(table.iloc[:, 0], errors="coerce")
        table = table.loc[outage_datetime.notna()].reset_index(drop=True)
        tables.append(table)

    return tables

In [4]:
workbook_paths = sorted(BPA_DATA_DIR.glob("*.xlsx"))
if not workbook_paths:
    raise FileNotFoundError(f"No Excel workbooks found in {BPA_DATA_DIR.resolve()}")

tables = []
for workbook_path in workbook_paths:
    workbook_tables = extract_transmission_line_tables(workbook_path)
    if not workbook_tables:
        print(f"[WARNING] {workbook_path.name}: table not found")
        continue

    tables.extend(workbook_tables)
    row_count = sum(len(table) for table in workbook_tables)
    print(f"[OK] {workbook_path.name}: {row_count:,} rows")

if not tables:
    raise ValueError(f"No {TABLE_TITLE!r} tables were found")

bpa_processed = pd.concat(tables, ignore_index=True, sort=False)
bpa_processed.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(bpa_processed):,} rows to {OUTPUT_PATH.resolve()}")
bpa_processed.head()

[OK] OutagesCY2014.xlsx: 5,389 rows
[OK] OutagesCY2015.xlsx: 5,474 rows
[OK] OutagesCY2016.xlsx: 4,719 rows
[OK] OutagesCY2017.xlsx: 5,114 rows
[OK] OutagesCY2018.xlsx: 4,602 rows
[OK] OutagesCY2019.xlsx: 5,002 rows
[OK] OutagesCY2020.xlsx: 4,788 rows
[OK] OutagesCY2021.xlsx: 4,974 rows
[OK] OutagesCY2022.xlsx: 2,157 rows
[OK] OutagesCY2023.xlsx: 1,743 rows
[OK] OutagesCY2024.xlsx: 3,765 rows
[OK] OutagesCY2025.xlsx: 4,136 rows
[OK] OutagesCY2026.xlsx: 2,536 rows
Saved 54,399 rows to /home/aj/phd/nyiso-data/src-bpa/bpa_processed.csv


,Out Datetime,In Datetime,Name,Voltage (kV),Line Type,Gen Flag,Length (miles),Duration (minutes),Outage Type,Cause Dispatch,...,Cause,Responsible System,O&M District,Transmission Owner NERC TADS,Out Datetime (PPT),In Datetime (PPT),Kilovolt,Megawatt Loss 1/,OMS Outage ID,OARS Outage ID
0,01/01/1988 11:00,NaN,Ashe-Marion No 2 500kV line,500.0,L,T,224.01,still out,Plan,Normally Out,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01/01/1988 11:00,NaN,Slatt tap to Ashe-Marion No 2 500kV line,500.0,T,,0.10,still out,Plan,Normally Out,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,01/20/1995 10:31,NaN,Satsop-Grays Harbor Public Development Authori...,230.0,L,,0.10,still out,Plan,Normally Out,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,04/03/1995 10:00,NaN,Bell-AVA Beacon No 2 115kV line,115.0,L,,6.20,still out,Plan,Normally Out,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,06/13/1996 10:20,NaN,Alvey-Fairview No 1 230kV line,230.0,L,T,97.46,still out,Plan,Normally Out,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
